### Домашнее задание Lite — вариант на PyTorch


In [1]:
import os
import zipfile
import random
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

%matplotlib inline

# Фиксация случайности для более стабильных результатов
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

Устройство: cpu


In [2]:
# Скачивание и распаковка базы
url = "https://storage.yandexcloud.net/aiueducation/Content/base/l3/hw_light.zip"
zip_path = "hw_light.zip"
extract_to = "/content"

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_to)

print("Архив распакован")

Архив распакован


In [3]:
# Путь к директории с базой
base_dir = "/content/hw_light"

# Списки для изображений и меток
x_train = []
y_train = []

img_height = 20
img_width = 20

# Сортировка нужна, чтобы порядок чтения был одинаковым между запусками
for patch in sorted(os.listdir(base_dir)):
    patch_path = os.path.join(base_dir, patch)

    for img_name in sorted(os.listdir(patch_path)):
        img_path = os.path.join(patch_path, img_name)

        # Чтение картинки в градациях серого и изменение размера до 20x20
        img = Image.open(img_path).convert("L").resize((img_width, img_height))
        img_array = np.array(img, dtype=np.float32)

        # PyTorch ожидает формат (каналы, высота, ширина)
        x_train.append(img_array[np.newaxis, :, :])

        # Метки классов сохраняем так же, как в исходном ноутбуке
        if patch == "0":
            y_train.append(0)
        elif patch == "3":
            y_train.append(1)
        else:
            y_train.append(2)

x_train = np.array(x_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.int64)

print("Размер массива x_train:", x_train.shape)
print("Размер массива y_train:", y_train.shape)

Размер массива x_train: (302, 1, 20, 20)
Размер массива y_train: (302,)


In [4]:
# Перевод данных в тензоры PyTorch
X_tensor = torch.tensor(x_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)

print("Форма тензора X_tensor:", X_tensor.shape)
print("Форма тензора y_tensor:", y_tensor.shape)

Форма тензора X_tensor: torch.Size([302, 1, 20, 20])
Форма тензора y_tensor: torch.Size([302])


In [5]:
results = []

hidden_units_list = [10, 100, 5000]
activation_list = ["relu", "linear"]
batch_size_list = [10, 100, 1000]

for hidden_units in hidden_units_list:
    for activation_name in activation_list:
        for batch_size in batch_size_list:

            # Создаем список слоев
            layers = [
                nn.Flatten(),
                nn.Linear(20 * 20, hidden_units)
            ]

            # Добавляем функцию активации
            if activation_name == "relu":
                layers.append(nn.ReLU())
            elif activation_name == "linear":
                pass
            else:
                raise ValueError(f"Неизвестная функция активации: {activation_name}")

            # Добавляем выходной слой
            layers.append(nn.Linear(hidden_units, 3))

            # Собираем модель
            model = nn.Sequential(*layers).to(device)

            # Функция потерь и оптимизатор
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=0.001)

            # Разбиваем данные на батчи
            loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

            # Обучение модели
            model.train()
            for epoch in range(10):
                for batch_x, batch_y in loader:
                    batch_x = batch_x.to(device)
                    batch_y = batch_y.to(device)

                    optimizer.zero_grad()
                    logits = model(batch_x)
                    loss = criterion(logits, batch_y)
                    loss.backward()
                    optimizer.step()

            # Оценка точности
            model.eval()
            with torch.no_grad():
                logits = model(X_tensor.to(device))
                predictions = torch.argmax(logits, dim=1)
                accuracy = (predictions == y_tensor.to(device)).float().mean().item()

            results.append([hidden_units, activation_name, batch_size, accuracy])

            print(
                f"Нейроны: {hidden_units:>4} | "
                f"Активация: {activation_name:>6} | "
                f"Batch size: {batch_size:>4} | "
                f"Точность: {accuracy:.4f}"
            )

Нейроны:   10 | Активация:   relu | Batch size:   10 | Точность: 0.3841
Нейроны:   10 | Активация:   relu | Batch size:  100 | Точность: 0.3675
Нейроны:   10 | Активация:   relu | Batch size: 1000 | Точность: 0.4106
Нейроны:   10 | Активация: linear | Batch size:   10 | Точность: 0.8609
Нейроны:   10 | Активация: linear | Batch size:  100 | Точность: 0.6788
Нейроны:   10 | Активация: linear | Batch size: 1000 | Точность: 0.5132
Нейроны:  100 | Активация:   relu | Batch size:   10 | Точность: 0.8576
Нейроны:  100 | Активация:   relu | Batch size:  100 | Точность: 0.7285
Нейроны:  100 | Активация:   relu | Batch size: 1000 | Точность: 0.5265
Нейроны:  100 | Активация: linear | Batch size:   10 | Точность: 0.8742
Нейроны:  100 | Активация: linear | Batch size:  100 | Точность: 0.5497
Нейроны:  100 | Активация: linear | Batch size: 1000 | Точность: 0.6391
Нейроны: 5000 | Активация:   relu | Batch size:   10 | Точность: 0.9106
Нейроны: 5000 | Активация:   relu | Batch size:  100 | Точность: